# MAS Validity Harness — Colab Pilot Runner (v2.1)

Run cells **top to bottom, in order**. There are two STOP points where you
report results back to Claude before continuing.

**Before you start (one time):**
1. Upload `mas_harness.zip` to a folder called `mas_harness` in your Google Drive (create it via drive.google.com).
2. Click the key icon (Secrets) in Colab's left sidebar → add secret `OPENROUTER_API_KEY` with your key → toggle "Notebook access" ON.


## Cell 1 — Mount Drive (survives disconnects)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ mas_harness
%ls

Mounted at /content/drive
/content/drive/MyDrive/ mas_harness
backbone.py            mas_harness.zip        scored_code_llama.jsonl
cp2_trajectory.txt     mas_loop.py            scored_math_llama.jsonl
data_sharded.json      metrics_baselines.py   scorers.py
embedder.py            metrics_geometry.py    STUDY_GUIDE.md
extract_features.py    metrics_klrd.py        test_synthetic.py
gauntlet.py            __pycache__/           trajs_code_llama.jsonl
lost_in_conversation/  README.md              trajs_math_llama.jsonl
mas_harness/           repair_elicitation.py  trajs_math_llama.jsonl.bak


## Cell 2 — One-time setup: unzip code, fetch benchmark data, install deps
(Re-running is harmless.)

In [2]:
!unzip -o -q mas_harness.zip
import os
if not os.path.exists('data_sharded.json'):
    rc = os.system('git clone --depth 1 https://github.com/microsoft/lost_in_conversation')
    rc = os.system('cp lost_in_conversation/data/sharded_instructions_600.json data_sharded.json')
    assert os.path.exists('data_sharded.json'), 'data fetch failed — rerun this cell'
!pip -q install tiktoken sentence-transformers
import json
d = json.load(open('data_sharded.json'))
from collections import Counter
print('records per task:', Counter(x['task'] for x in d))

records per task: Counter({'data2text': 120, 'database': 107, 'actions': 105, 'math': 103, 'code': 100, 'summary': 92})


In [3]:
!python scorers.py trajs_code_llama.jsonl scored_code_llama.jsonl

['one']
['one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
[]
[]
[0, 2, 3, 24, 33]
False
{'b': 4}
['the number of odd elements 4n the str7g 7 of the 7nut.']
2
e
Error executing code: Input string must contain only English letters
[1000, 750, 500]
140
[1, 5]
(0, 1)
Product of signs: 0
Sum of absolute values: 13
None
<string>:59: SyntaxWarning: invalid escape sequence '\.'
False
True
False
False
False
False
The Brazilian factorial of 4 is: 288
my_class.AA
string
[11, 2]
[2, 2, 2]
Is the product equal to the input?  True
Error executing code: invalid syntax (<string>, line 64)
1
No special number found
22
[]
1, 2, 3, 4, 5
Enter the number: Error executing code: EOF when reading a line
Derivative Coefficients: [3, 2, 6, 16, 0]
Derivative Polynomial: 3x^4 + 2x^4 + 6x^4 + 16x^4 + 
7
Enter side length 1 (a): Error executing code: EOF when reading a line
Error executing code: invalid syntax (<string>, line 64)
[]
False
True
True
(15, 120)
(0, 1)
90
Error executing code: C

## Cell 3 — API key from Colab Secrets (never paste the key in a cell)

In [3]:
import os
from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('key loaded:', os.environ['OPENROUTER_API_KEY'][:8] + '...')

key loaded: sk-or-v1...


In [4]:
!python repair_elicitation.py trajs_math_llama.jsonl data_sharded.json
!python scorers.py trajs_math_llama.jsonl scored_math_llama.jsonl

repaired sharded-GSM8K/1190: parseable=True (was: ... the third day, which aligns with the remaining 78 bandages.)
repaired sharded-GSM8K/215: parseable=True (was: ...inputs.  The estimated total hours Lori needs to work is 44.)
repaired sharded-GSM8K/740: parseable=True (was: ...ce the number of students interested in playing video games.)
repaired sharded-GSM8K/346: parseable=True (was: ...s/week, I estimate that Jenny eats 36 - 27 = 9 cookies/week.)
repaired sharded-GSM8K/369: parseable=True (was: ...198  The original budget was $500/gift x 44 gifts = $22,000.)
repaired sharded-GSM8K/40: parseable=True (was: ... = 8 years old  **ANSWER:** Brandon's iPhone is 8 years old.)
repaired sharded-GSM8K/751: parseable=True (was: ...tity of bananas in Gunther's pile (41) in the previous turn.)

7/20 records re-elicited; backup at trajs_math_llama.jsonl.bak
scored 20 trajectories, success rate = 9/20 = 0.45


## Cell 4 — CHECKPOINT 1: instrument calibration (no API cost)
Must print **14/14 checks passed**. Copy the FULL output of this cell for Claude.
If anything says FAIL → **STOP**, send the output, do not continue.

In [14]:
!python test_synthetic.py

agent     vcv_mean  selfrep briefcos  novelty klrd_slope   klrd_EXC rolestab_sl   lz76  top1var
frozen       0.000    1.000    0.278    0.125     0.2903     0.0000     -0.0000   0.80    1.000
random       1.182    0.300    0.041    0.312     0.1603     0.0083      0.0003   1.60    0.257
parrot       0.000    1.000    1.000    0.125     0.3503     0.0000     -0.0000   0.80    1.000
drifter      1.166    0.313    0.009    0.375     0.2377     0.0988     -0.0878   2.01    0.303

--- assertions ---
PASS  frozen VCV ~ 0
PASS  frozen self-repetition ~ 1
PASS  frozen raw KLRD slope LARGE (the artifact!)
PASS  frozen KLRD-EXCESS ~ 0 (fix works)
PASS  random KLRD-EXCESS ~ 0 (stationary)
PASS  random VCV high
PASS  random novelty > frozen
PASS  random KLRD slope > 0
PASS  parrot brief-cosine ~ 1
PASS  parrot VCV ~ 0
PASS  drifter KLRD-EXCESS > 0 (real drift)
PASS  drifter EXCESS > frozen EXCESS
PASS  drifter rolestab slope < 0
PASS  frozen top1_var >= drifter

14/14 checks passed


## Cell 5 — Pilot: 20 sharded math records on llama-3.1-8b (~$0.10-0.30, ~5-10 min)
Resumable: if it crashes or Colab disconnects, just re-run this cell — completed records are skipped.

In [21]:
import json
from backbone import ChatBackbone
from mas_loop import run_many

d = json.load(open('data_sharded.json'))
recs = [x for x in d if x['task'] == 'math'][:20]
bb = ChatBackbone(provider="openrouter")   # meta-llama/llama-3.1-8b-instruct
run_many(recs, bb, 'trajs_math_llama.jsonl', sleep_s=0.3)
print('\nDone. Trajectories saved to Drive.')

[0] done sharded-GSM8K/1246 (4 shards)
[1] done sharded-GSM8K/1166 (7 shards)
[2] done sharded-GSM8K/158 (8 shards)
[3] done sharded-GSM8K/435 (4 shards)
[4] done sharded-GSM8K/1190 (8 shards)
[5] done sharded-GSM8K/14 (5 shards)
[6] done sharded-GSM8K/237 (5 shards)
[7] done sharded-GSM8K/215 (4 shards)
[8] done sharded-GSM8K/621 (6 shards)
[9] done sharded-GSM8K/968 (9 shards)
[10] done sharded-GSM8K/739 (6 shards)
[11] done sharded-GSM8K/477 (5 shards)
[12] done sharded-GSM8K/740 (6 shards)
[13] done sharded-GSM8K/1027 (6 shards)
[14] done sharded-GSM8K/991 (4 shards)
[15] done sharded-GSM8K/346 (8 shards)
[16] done sharded-GSM8K/369 (7 shards)
[17] done sharded-GSM8K/40 (4 shards)
[18] done sharded-GSM8K/751 (7 shards)
[19] done sharded-GSM8K/307 (4 shards)

Done. Trajectories saved to Drive.


## Cell 6 — Score outcomes (math = safe number parsing, no sandbox needed)

In [22]:
!python scorers.py trajs_math_llama.jsonl scored_math_llama.jsonl

scored 20 trajectories, success rate = 8/20 = 0.40


## Cell 7 — CHECKPOINT 2: build the report for Claude
This prints (a) the success rate, (b) per-record results, (c) ONE full trajectory,
and also saves the trajectory to `cp2_trajectory.txt` in Drive.
**STOP after this cell.** Send Claude: Cell 4 output + Cell 7 output (or the txt file).
Before sending, read the trajectory yourself and note: did the Planner track
constraints? Did the Executor end with FINAL ANSWER? Did the Critic catch anything?

In [23]:
import json

trajs = [json.loads(l) for l in open('scored_math_llama.jsonl')]
n = len(trajs); ok = sum(t['outcome'] for t in trajs)
print(f'=== CP2 REPORT ===')
print(f'records: {n} | successes: {ok} | success rate: {ok/n:.2f}')
print(f'(Gate 1: need 0.2-0.8 to proceed)\n')
for t in trajs:
    print(f"  {t['task_id']}: shards={t['n_shards']} outcome={t['outcome']}")

# one full trajectory: pick a failure if any exists (more informative), else first
pick = next((t for t in trajs if t['outcome'] == 0), trajs[0])
lines = [f"TASK {pick['task_id']} | outcome={pick['outcome']} | model={pick['model']}",
         f"GOLD: {pick['record_meta'].get('answer','')[-120:]}", '']
for turn in pick['turns']:
    lines.append(f"--- shard {turn['shard_idx']} | {turn['role'].upper()} ---")
    lines.append(turn['text']); lines.append('')
report = '\n'.join(lines)
open('cp2_trajectory.txt', 'w').write(report)
print('\n' + report[:4000])
print('\n[full trajectory saved to cp2_trajectory.txt in Drive]')

=== CP2 REPORT ===
records: 20 | successes: 8 | success rate: 0.40
(Gate 1: need 0.2-0.8 to proceed)

  sharded-GSM8K/1246: shards=4 outcome=1
  sharded-GSM8K/1166: shards=7 outcome=0
  sharded-GSM8K/158: shards=8 outcome=0
  sharded-GSM8K/435: shards=4 outcome=1
  sharded-GSM8K/1190: shards=8 outcome=0
  sharded-GSM8K/14: shards=5 outcome=0
  sharded-GSM8K/237: shards=5 outcome=0
  sharded-GSM8K/215: shards=4 outcome=1
  sharded-GSM8K/621: shards=6 outcome=0
  sharded-GSM8K/968: shards=9 outcome=0
  sharded-GSM8K/739: shards=6 outcome=1
  sharded-GSM8K/477: shards=5 outcome=1
  sharded-GSM8K/740: shards=6 outcome=0
  sharded-GSM8K/1027: shards=6 outcome=1
  sharded-GSM8K/991: shards=4 outcome=0
  sharded-GSM8K/346: shards=8 outcome=1
  sharded-GSM8K/369: shards=7 outcome=0
  sharded-GSM8K/40: shards=4 outcome=1
  sharded-GSM8K/751: shards=7 outcome=0
  sharded-GSM8K/307: shards=4 outcome=0

TASK sharded-GSM8K/1166 | outcome=0 | model=meta-llama/llama-3.1-8b-instruct
GOLD: , an avocado

In [5]:
import json, re
from scorers import parse_gold_math, parse_pred_math

trajs = [json.loads(l) for l in open('scored_math_llama.jsonl')]
print(f"{'task_id':28s} {'out':3s} {'gold':>8s} {'pred':>8s}  FA-line?  ")
for t in trajs:
    gold = parse_gold_math(t['record_meta'].get('answer',''))
    pred = parse_pred_math(t['final_answer'])
    has_fa = bool(re.search(r'FINAL ANSWER', t['final_answer'], re.I))
    flag = '  <-- INSPECT' if (t['outcome']==0 and (pred is None or not has_fa)) else ''
    print(f"{t['task_id']:28s} {t['outcome']:3d} {str(gold):>8s} {str(pred):>8s}  {has_fa}{flag}")

print("\n--- final-turn tails of flagged failures ---")
for t in trajs:
    pred = parse_pred_math(t['final_answer'])
    has_fa = bool(re.search(r'FINAL ANSWER', t['final_answer'], re.I))
    if t['outcome']==0 and (pred is None or not has_fa):
        print(f"\n### {t['task_id']} | gold={parse_gold_math(t['record_meta'].get('answer',''))}")
        print(t['final_answer'][-300:])

task_id                      out     gold     pred  FA-line?  
sharded-GSM8K/1246             1     3360     3360  True
sharded-GSM8K/1166             0     2350     1670  True
sharded-GSM8K/158              1       34    34.00  True
sharded-GSM8K/435              1        5        5  True
sharded-GSM8K/1190             0       19      100  True
sharded-GSM8K/14               0       60       50  True
sharded-GSM8K/237              0       90      105  True
sharded-GSM8K/215              1       44       44  True
sharded-GSM8K/621              0     3000     4300  True
sharded-GSM8K/968              0      227     4.85  True
sharded-GSM8K/739              1       98       98  True
sharded-GSM8K/477              1       20       20  True
sharded-GSM8K/740              0       25       13  True
sharded-GSM8K/1027             1       12       12  True
sharded-GSM8K/991              0       15     1051  True
sharded-GSM8K/346              1        9        9  True
sharded-GSM8K/369        

---
## LOCKED until Claude reviews CP1+CP2
### Cell 8 — Scale up (full math family + code family)

In [6]:
# UNLOCK ONLY AFTER CHECKPOINT REVIEW
RUN = True
if RUN:
    import json, os
    from backbone import ChatBackbone
    from mas_loop import run_many
    d = json.load(open('data_sharded.json'))
    bb = ChatBackbone(provider="openrouter")
    run_many([x for x in d if x['task']=='math'], bb, 'trajs_math_llama.jsonl', sleep_s=0.3)
    run_many([x for x in d if x['task']=='code'], bb, 'trajs_code_llama.jsonl', sleep_s=0.3)
    os.system('python scorers.py trajs_math_llama.jsonl scored_math_llama.jsonl')
    # code scoring EXECUTES model-generated code — Colab VM is disposable, acceptable
    os.system('python scorers.py trajs_code_llama.jsonl scored_code_llama.jsonl')

[20] done sharded-GSM8K/941 (6 shards)
[21] done sharded-GSM8K/122 (5 shards)
[22] done sharded-GSM8K/1124 (6 shards)
[23] done sharded-GSM8K/976 (8 shards)
[24] done sharded-GSM8K/965 (8 shards)
[25] done sharded-GSM8K/510 (5 shards)
[26] done sharded-GSM8K/189 (6 shards)
[27] done sharded-GSM8K/1180 (7 shards)
[28] done sharded-GSM8K/598 (6 shards)
[29] done sharded-GSM8K/144 (12 shards)
[30] done sharded-GSM8K/378 (6 shards)
[31] done sharded-GSM8K/1068 (5 shards)
[32] done sharded-GSM8K/44 (5 shards)
[33] done sharded-GSM8K/1240 (6 shards)
[34] done sharded-GSM8K/359 (7 shards)
[35] done sharded-GSM8K/543 (4 shards)
[36] done sharded-GSM8K/178 (4 shards)
[37] done sharded-GSM8K/961 (5 shards)
[38] done sharded-GSM8K/1260 (7 shards)
[39] done sharded-GSM8K/420 (5 shards)
[40] done sharded-GSM8K/698 (5 shards)
[41] done sharded-GSM8K/234 (6 shards)
[42] done sharded-GSM8K/416 (6 shards)
[43] done sharded-GSM8K/1197 (5 shards)
[44] done sharded-GSM8K/465 (8 shards)
[45] done sharded-G

In [7]:
import json, re
trajs = [json.loads(l) for l in open('scored_code_llama.jsonl')]
n = len(trajs); ok = sum(t['outcome'] for t in trajs)
no_block = [t['task_id'] for t in trajs if t['outcome']==0
            and not re.search(r'```(python)?\s*\n', t['final_answer'])]
print(f'code: {ok}/{n} = {ok/n:.2f} success')
print(f'failures with NO code block (artifact suspects): {len(no_block)}')
print(no_block[:10])

code: 0/100 = 0.00 success
failures with NO code block (artifact suspects): 0
[]


### Cell 9 — Features + Gauntlet (needs scale-up done; GPU runtime makes embedding faster)

In [3]:
# UNLOCK ONLY AFTER SCALE-UP
RUN = True
if RUN:
    import os
    os.system('cat scored_math_llama.jsonl scored_code_llama.jsonl > scored_all_llama.jsonl')
    os.system('python extract_features.py scored_all_llama.jsonl features_llama.csv sbert')
    os.system('python gauntlet.py features_llama.csv 2000')
    # results also saved as gauntlet_<family>_{univariate,incremental}.csv in Drive

In [5]:
import pandas as pd

feat = pd.read_csv('features_llama.csv')
feat["family"] = feat["task"]
is_code = feat["task"] == "code"
feat.loc[is_code & feat["task_id"].str.contains("HumanEval"), "family"] = "code-HumanEval"
feat.loc[is_code & feat["task_id"].str.contains("livecodebench"), "family"] = "code-LCB"

for fam in ['math', 'code-HumanEval', 'code-LCB']:
    d = feat[feat["family"] == fam]
    print(f"\n{'='*70}\n=== {fam}: n={len(d)}, success rate={d['outcome'].mean():.2f} ===")
    try:
        inc = pd.read_csv(f'gauntlet_{fam}_incremental.csv')
        base_auc = (inc['auc_with'] - inc['delta_auc']).iloc[0]
        print(f"baselines-only AUC = {base_auc:.3f}")
        print("\n--- INCREMENTAL (the verdict) ---")
        print(inc.to_string(index=False))
        uni = pd.read_csv(f'gauntlet_{fam}_univariate.csv')
        print("\n--- top 10 univariate (descriptive) ---")
        print(uni.head(10).to_string(index=False))
    except FileNotFoundError:
        print("(no gauntlet files — benched by the guard, as expected for LCB)")


=== math: n=103, success rate=0.42 ===
baselines-only AUC = 0.681

--- INCREMENTAL (the verdict) ---
                   candidate  auc_with  delta_auc  delta_sd   perm_p        tier     bh_q  survives
       planner_rolestab_mean  0.718159   0.037403  0.012799      NaN exploratory 1.000000     False
      executor_rolestab_mean  0.709167   0.028411  0.015138      NaN exploratory 1.000000     False
        planner_rolestab_min  0.706705   0.025950  0.015173      NaN exploratory 1.000000     False
  executor_klrd_excess_slope  0.693430   0.012674  0.006992 0.138931     PRIMARY 0.185407     False
    critic_klrd_excess_slope  0.690368   0.009612  0.010671 0.183408     PRIMARY 0.185407     False
   planner_klrd_excess_slope  0.689554   0.008798  0.008362 0.185407     PRIMARY 0.185407     False
         executor_klrd_slope  0.687481   0.006725  0.013142      NaN exploratory 1.000000     False
     executor_rolestab_slope  0.685523   0.004767  0.010652      NaN exploratory 1.000000     Fals

In [6]:
import shutil, os, hashlib, json, datetime
os.makedirs('results_v1_llama', exist_ok=True)
keep = ['features_llama.csv',
        'gauntlet_math_incremental.csv','gauntlet_math_univariate.csv',
        'gauntlet_code-HumanEval_incremental.csv','gauntlet_code-HumanEval_univariate.csv',
        'scored_math_llama.jsonl','scored_code_llama.jsonl',
        'trajs_math_llama.jsonl','trajs_code_llama.jsonl']
manifest = {}
for f in keep:
    if os.path.exists(f):
        shutil.copy(f, f'results_v1_llama/{f}')
        manifest[f] = {'sha256_16': hashlib.sha256(open(f,'rb').read()).hexdigest()[:16],
                       'bytes': os.path.getsize(f)}
manifest['_meta'] = {
    'frozen': str(datetime.datetime.now()),
    'backbone': 'meta-llama/llama-3.1-8b-instruct, temperature 1.0',
    'protocol': 'scorer v3 (official LiC evaluator), temp-0 elicitation + strict retry, '
                'role-tag-stripped features, gauntlet v2.1 (repeated 5-fold, two-tier BH-FDR, '
                'pre-registered PRIMARY = klrd_excess_slope per role)',
    'headline': 'Complete null: no candidate survives. Baselines-only AUC math=0.681. '
                'Best exploratory: planner_rolestab_mean rho=0.470 (promoted to PRIMARY for study 2).'}
json.dump(manifest, open('results_v1_llama/MANIFEST.json','w'), indent=1)
print('frozen:', list(manifest.keys()))

frozen: ['features_llama.csv', 'gauntlet_math_incremental.csv', 'gauntlet_math_univariate.csv', 'gauntlet_code-HumanEval_incremental.csv', 'gauntlet_code-HumanEval_univariate.csv', 'scored_math_llama.jsonl', 'scored_code_llama.jsonl', 'trajs_math_llama.jsonl', 'trajs_code_llama.jsonl', '_meta']


In [7]:
!cp -r pilot_log results_v1_llama/pilot_log

cp: cannot stat 'pilot_log': No such file or directory


In [8]:
import os, shutil
os.makedirs('results_v1_llama/pilot_log', exist_ok=True)

# scattered pilot artifacts that may exist at top level
for f in ['cp2_trajectory.txt', 'trajs_math_llama.jsonl.bak']:
    if os.path.exists(f):
        shutil.copy(f, f'results_v1_llama/pilot_log/{f}')
        print('saved:', f)
    else:
        print('MISSING:', f)

# regenerate the final math parse-audit as a real file (it only ever lived in cell output)
import json, re
from scorers import parse_gold_math, parse_pred_math
lines = [f"{'task_id':28s} {'out':3s} {'gold':>8s} {'pred':>8s}  FA-line?"]
for t in [json.loads(l) for l in open('scored_math_llama.jsonl')]:
    gold = parse_gold_math(t['record_meta'].get('answer',''))
    pred = parse_pred_math(t['final_answer'])
    fa = bool(re.search(r'FINAL ANSWER', t['final_answer'], re.I))
    lines.append(f"{t['task_id']:28s} {t['outcome']:3d} {str(gold):>8s} {str(pred):>8s}  {fa}")
open('results_v1_llama/pilot_log/final_parse_audit_math.txt','w').write('\n'.join(lines))
print('\naudit regenerated for all', len(lines)-1, 'math records')

saved: cp2_trajectory.txt
saved: trajs_math_llama.jsonl.bak

audit regenerated for all 103 math records


In [9]:
gitignore = """lost_in_conversation/
__pycache__/
mas_harness.zip
data_sharded.json
sample_data/
"""
open('.gitignore','w').write(gitignore)

82

In [10]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
USER, REPO = 'taifONduty', 'mas-validity-harness'
!git init -q
!git config user.name "Taif Ahmed Turjo" && git config user.email "ahmedtaif437@gmail.com"
!git add -A
!git commit -q -m "v1: calibrated harness (14/14), KLRD-excess estimator, llama-3.1-8b study (203 records), gauntlet v2.1 null result"
!git branch -M main
import os; os.system(f'git push -q https://{token}@github.com/{USER}/{REPO}.git main')
print('pushed')

pushed


In [11]:
# P1 + P3
!rm -rf mas_harness
!rm -f scored_all_llama.jsonl cp2_trajectory.txt trajs_math_llama.jsonl.bak

# P4: pre-register study-2 primaries (stem-level: tests the construct per role, m=6 total)
s = open('gauntlet.py').read()
s = s.replace('PRIMARY_STEMS = ["klrd_excess_slope"]',
  'PRIMARY_STEMS = ["klrd_excess_slope", "rolestab_mean"]  '
  '# rolestab added 2026-07 BEFORE DeepSeek replication, per study-1 exploratory promotion (planner expected locus)')
open('gauntlet.py','w').write(s)
print('rolestab_mean' in open('gauntlet.py').read())

True
